# Group Assignment: Prediction in Bioinformatics

**Fundamentals of Health Data Science**

In the individual assignment you followed a guided path. **This time there is no path: your group designs it.**

Your group has been allocated **one** of three problems:

| | Problem | Question |
|---|---|---|
| **A** | Antimicrobial resistance from bacterial genomes | Is this *E. coli* isolate resistant to ciprofloxacin? |
| **B** | Blood gene expression to detect active TB | Does this patient have active TB? |
| **C** | Malaria parasites that clear slowly | Will this infection clear slowly after artemisinin treatment? |

Before you start, read your problem's section in **Group Assignment: the three problems and their data dictionaries**.

> All datasets are **simulated for teaching**. No real patients or isolates are included.

### Rules: read before you start

**Group work**
- One notebook per group. Everyone in the group is responsible for **all** of it.
- Every member must be able to **explain and defend any part** of the notebook, not only the part they led.
- Any member may be asked to complete a **short timed test or oral check** on any part of the notebook.

**Use of AI tools**
- Your plan, your decisions and your conclusions must be **in your own words. Do not use AI tools to write them.**
- **Any use of AI tools must be stated** in each member's declaration: what it was used for, and where.

**Deadline:** **Monday 16 November 2026**. You may submit earlier. **How it is marked:** details will be shared separately.

**What you hand in:** this notebook, run from top to bottom (*Restart and run all*), with the contribution statement and every member's declaration completed. Name the file with your group number and problem, e.g. `G03_ProblemC.ipynb`.

---
## 1. Your group and your data (PROVIDED)

Fill in the three settings, then run the cell.
- `MEMBERS`: every member's registration number, **exactly as on the class list**. For a group of two, leave the third as `""`.
- `GROUP_ID`: your group number from the allocation list, e.g. `"G03"`.
- `PROBLEM`: your allocated problem: `"A"`, `"B"` or `"C"`.

Keep your problem's data file in the same folder as this notebook (on Google Colab, the cell will ask you to upload it). **Write the group fingerprint into every member's declaration.**

In [ ]:
# ===================== PROVIDED CELL: fill in the three settings, do not edit the rest =====================
MEMBERS = ["", "", ""]   # registration numbers, exactly as on the class list
GROUP_ID = ""                   # e.g. "G03"
PROBLEM = ""                    # "A", "B" or "C"

import os, hashlib
import numpy as np
import pandas as pd

FILES = {"A": "problemA_ecoli_cipro_genomes.csv", "B": "problemB_tb_blood_expression.csv", "C": "problemC_malaria_clearance.csv"}
ID_COLUMN = {"A": "isolate_id", "B": "sample_id", "C": "patient_id"}
SAMPLE_SIZE = {"A": 2000, "B": 1500, "C": 2500}
OUTCOME = {"A": "cipro_resistant", "B": "active_tb", "C": "slow_clearance"}

members = sorted(m.strip().upper().replace(" ", "") for m in MEMBERS if m.strip())
problem = PROBLEM.strip().upper()
if not 2 <= len(members) <= 3:
    raise ValueError("Type the registration numbers of all 2 or 3 group members in MEMBERS.")
if len(set(members)) < len(members):
    raise ValueError("The same registration number is typed twice in MEMBERS.")
if problem not in FILES:
    raise ValueError('Set PROBLEM to "A", "B" or "C" (your allocated problem).')
if not GROUP_ID.strip():
    raise ValueError('Set GROUP_ID to your group number from the allocation list, e.g. "G03".')

DATA_FILE = FILES[problem]
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files          # only on Google Colab
        print(f"Please upload {DATA_FILE}"); files.upload()
    except ImportError:
        raise FileNotFoundError(f"Put {DATA_FILE} in the same folder as this notebook.")

GROUP_SEED = int(hashlib.sha256(("|".join(members) + "|" + problem).encode()).hexdigest(), 16) % (2**31)
SPLIT_SEED = GROUP_SEED % 10000

everyone = pd.read_csv(DATA_FILE)
all_ids = np.sort(everyone[ID_COLUMN[problem]].unique())
our_ids = np.random.default_rng(GROUP_SEED).choice(all_ids, SAMPLE_SIZE[problem], replace=False)
our_data = (everyone[everyone[ID_COLUMN[problem]].isin(our_ids)]
            .sample(frac=1, random_state=SPLIT_SEED).reset_index(drop=True))

fp = hashlib.sha256(",".join(sorted(our_ids)).encode()).hexdigest()[:8].upper()
GROUP_FINGERPRINT = fp[:4] + "-" + fp[4:]

print(f"Group {GROUP_ID.strip().upper()} | Problem {problem} | members: {', '.join(members)}")
print(f"Group fingerprint : {GROUP_FINGERPRINT}   <- write this in every member's declaration")
print(f"SPLIT_SEED        : {SPLIT_SEED}   <- use random_state=SPLIT_SEED whenever you shuffle or split")
print(f"Your data         : {our_data.shape[0]} rows, {our_data.shape[1]} columns, in a table called our_data")
print(f"Outcome column    : {OUTCOME[problem]}")

## 2. Libraries

A starting set. Add any others you need, here at the top.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, accuracy_score, recall_score, precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.calibration import CalibrationDisplay
import joblib

---
## 3. Expected stages

A modelling project usually passes through these stages. **You may add, merge, re-order or skip stages**, as long as your plan explains why.

1. Understand the problem and the data dictionary (who will use the model, and when?)
2. First look at the data, and cleaning rules
3. Leakage check (what is known at the moment of prediction?)
4. Train/test split
5. Exploration, on the training data only
6. A baseline model
7. Feature work: transformations, interactions, encoding, choice of columns
8. Pipeline and cross-validation
9. The outcome rate, and the choice of cut-off (if relevant to your problem)
10. Overfitting check and tuning
11. Final test, once
12. Save the model, write a model card, and answer the question

> Your problem has **at least one challenge you have not met in the demonstration notebook**. Look for it in the data dictionary and in your exploration.

---
## 4. Your plan (complete before you write any code)

For each stage you will carry out, say **what** you will do, **why**, and **who leads** it. You may update the plan as you go: if you change it, add a line saying what changed and why.

| Stage | What we will do | Why | Led by |
|---|---|---|---|
| | | | |
| | | | |
| | | | |
| | | | |

**What we think the main challenge in our problem is:**

**Changes to the plan as we went (and why):** 

---
## 5. Your work

Add as many cells as you need. For each stage, a short markdown cell with **Where we are · Why this step · What we do · What we decided** (as in the demonstration notebook) makes your reasoning easy to follow and to mark.

In [ ]:
# Write your code here

---
## 6. Conclusions

**The answer to the question, for the user of your model** (in plain language, quoting your results):

**How well does your model work on new data, and how sure are you?**

**Limitations, and what would need to happen before this model could be used for real:** 

---
## 7. Contribution statement

| Member (name) | Registration number | What I led | What I reviewed or checked |
|---|---|---|---|
| | | | |
| | | | |
| | | | |

---
## 8. Declarations (one per member)

### Member 1
1. **Name and registration number:**
2. **Group number, problem and group fingerprint:**
3. **I have read through all the code in this notebook, and it represents what our group intended to do. (yes / no):**
4. **I can explain and defend any part of this notebook, not only the parts I led. (yes / no):**
5. **The written parts I contributed are in my own words, without AI tools. (yes / no):**
6. **Use of AI tools:** I used AI tools for *(write "none", or describe exactly what you used them for, and where)*:
7. I understand that I may be asked to complete a **short timed test or oral check** on any part of this work. **(yes / no):**
8. **Date:**

### Member 2
1. **Name and registration number:**
2. **Group number, problem and group fingerprint:**
3. **I have read through all the code in this notebook, and it represents what our group intended to do. (yes / no):**
4. **I can explain and defend any part of this notebook, not only the parts I led. (yes / no):**
5. **The written parts I contributed are in my own words, without AI tools. (yes / no):**
6. **Use of AI tools:** I used AI tools for *(write "none", or describe exactly what you used them for, and where)*:
7. I understand that I may be asked to complete a **short timed test or oral check** on any part of this work. **(yes / no):**
8. **Date:**

### Member 3 (leave blank for a group of two)
1. **Name and registration number:**
2. **Group number, problem and group fingerprint:**
3. **I have read through all the code in this notebook, and it represents what our group intended to do. (yes / no):**
4. **I can explain and defend any part of this notebook, not only the parts I led. (yes / no):**
5. **The written parts I contributed are in my own words, without AI tools. (yes / no):**
6. **Use of AI tools:** I used AI tools for *(write "none", or describe exactly what you used them for, and where)*:
7. I understand that I may be asked to complete a **short timed test or oral check** on any part of this work. **(yes / no):**
8. **Date:** 